# Project Wayne-Stark: Multi-Dimensional Urban Triage
**Architecture:** Materialized View Pivoting (EAV to Dimensional) -> PySpark Join & Scoring -> Concurrent Gemini 3.8 Flash Allocation -> Executive AI Synthesis

### Problem Statement:
The source dataset (`us_cities_demographics.us_cities_demographics.big_cities_demographic_indicators`) stores indicators in an Entity-Attribute-Value (EAV / long) format. Querying this raw structure directly for cross-metric analysis incurs heavy compute penalties.

### Solution:
1. **Dimensional Modeling via Delta Tables:** Pivot raw key-value pairs into domain-isolated wide tables (Wayne: Economic Distress, Stark: Health & Infrastructure, Joint: Vulnerability).
2. **Deterministic Pre-Filtering:** Join the MVs in PySpark and calculate an `Intervention_Priority_Score` to isolate the top 3 hot spots.
3. **Multi-Agent Synthesis:** Run concurrent REST API calls to `gemini-3.8-flash` simulating J.A.R.V.I.S. and Batcomputer co-analysis, capped at 4 total calls.

## 1. Dimensional Modeling: Pivoting EAV to Domain-Specific Views
**Goal:** Reshape long-format Entity-Attribute-Value (EAV) data into cached, domain-specific wide tables.

The raw John Snow Labs dataset stores demographic indicators as key-value pairs. Querying this EAV structure directly for multi-variable analysis requires massive compute overhead at runtime. To optimize this, we build **Delta Tables** that pivot the rows into columns while filtering for `Gender = 'Both'` and `Race_Ethnicity = 'Total'` to prevent duplicating segmented demographic groups.

By physically separating the views into domains (Wayne Foundation: Economic/Education vs. Stark Industries: Health/Infrastructure), we establish clean governance boundaries and ensure that downstream applications only query the specific data marts they need.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.wayne_stark;
USE CATALOG workspace;
USE SCHEMA wayne_stark;

-- 1. Wayne Foundation Domain: Economic & Educational Distress
CREATE OR REPLACE TABLE workspace.wayne_stark.econ_distress AS
SELECT 
    Place AS city, 
    State_Abbreviation AS state,
    MAX(CASE WHEN Demographic_Indicator = 'Median Household Income (Dollars)' THEN Value END) AS median_household_income,
    MAX(CASE WHEN Demographic_Indicator = 'Percent Unemployed' THEN Value END) AS percent_unemployed,
    MAX(CASE WHEN Demographic_Indicator = 'Percent Living Below 200% Poverty Level' THEN Value END) AS percent_poverty_level,
    MAX(CASE WHEN Demographic_Indicator = 'Percent of Households Whose Housing Costs Exceed 35% of Income' THEN Value END) AS percent_high_housing_costs,
    MAX(CASE WHEN Demographic_Indicator = 'Percent of High School Graduates (Over Age 18)' THEN Value END) AS percent_hs_graduates
FROM us_cities_demographics.us_cities_demographics.big_cities_demographic_indicators
WHERE Gender = 'Both' AND Race_Ethnicity = 'All' 
GROUP BY Place, State_Abbreviation;

-- 2. Stark Industries Domain: Systemic Health & Life Expectancy
CREATE OR REPLACE TABLE workspace.wayne_stark.systemic_health_deficits AS
SELECT 
    Place AS city, 
    State_Abbreviation AS state,
    MAX(CASE WHEN Demographic_Indicator = 'Percent of Population Uninsured' THEN Value END) AS percent_uninsured,
    MAX(CASE WHEN Demographic_Indicator = 'Infant Mortality Rate (Per 1,000 live Birth)' THEN Value END) AS infant_mortality_rate,
    MAX(CASE WHEN Demographic_Indicator = 'Life Expectancy at Birth (Years)' THEN Value END) AS life_expectancy
FROM us_cities_demographics.us_cities_demographics.big_cities_demographic_indicators
WHERE Gender = 'Both' AND Race_Ethnicity = 'All'
GROUP BY Place, State_Abbreviation;

-- 3. Joint Domain: Vulnerable Populations & Youth
CREATE OR REPLACE TABLE workspace.wayne_stark.vulnerable_demographics AS
SELECT 
    Place AS city, 
    State_Abbreviation AS state,
    MAX(CASE WHEN Demographic_Indicator = 'Total population (People)' THEN Value END) AS total_population,
    MAX(CASE WHEN Demographic_Indicator = 'Percent of Population Under 18' THEN Value END) AS percent_under_18,
    MAX(CASE WHEN Demographic_Indicator = 'Percent of Children Living in Poverty' THEN Value END) AS percent_children_in_poverty,
    MAX(CASE WHEN Demographic_Indicator = 'Percent of 3 and 4 Year Olds Currently Enrolled in Preschool' THEN Value END) AS percent_preschool_enrollment
FROM us_cities_demographics.us_cities_demographics.big_cities_demographic_indicators
WHERE Gender = 'Both' AND Race_Ethnicity = 'All'
GROUP BY Place, State_Abbreviation;

num_affected_rows,num_inserted_rows


## 2. The Grand Join & Inverted Metric Scoring
**Goal:** Join the dimensional views, normalize the metrics, and calculate a deterministic `Intervention_Priority_Score`.

With the data pivoted and cached by Databricks SQL, we use PySpark to join the tables and identify the top 3 critical municipalities. 

**Engineering Business Logic:**
Building a composite urgency index requires all metrics to scale in the same direction (higher score = more urgent). While metrics like *Child Poverty* naturally fit this, education metrics like *High School Graduation Rate* and *Preschool Enrollment* are positive metrics. 

To maintain mathematical integrity, we invert these positive indicators into **deficits** (e.g., `100.0 - Graduation Rate`). We also wrap all calculations in `coalesce(col, 100.0)` to guarantee that missing values do not artificially inflate a city's urgency score.

In [0]:
from pyspark.sql.functions import col, desc, coalesce, lit

# 1. Load the wide Delta tables
econ_df = spark.table("workspace.wayne_stark.econ_distress")
health_df = spark.table("workspace.wayne_stark.systemic_health_deficits")
vuln_df = spark.table("workspace.wayne_stark.vulnerable_demographics")

# 2. Join into a unified analytical dataset
joint_df = econ_df.join(health_df, ["city", "state"]) \
                  .join(vuln_df, ["city", "state"])

# 3. Handle Nulls and Invert Positive Metrics
# Subtracting graduation/enrollment from 100 to create "deficit" percentages
scored_df = joint_df.withColumn("child_poverty", coalesce(col("percent_children_in_poverty"), lit(0.0))) \
                    .withColumn("uninsured", coalesce(col("percent_uninsured"), lit(0.0))) \
                    .withColumn("high_housing", coalesce(col("percent_high_housing_costs"), lit(0.0))) \
                    .withColumn("hs_deficit", lit(100.0) - coalesce(col("percent_hs_graduates"), lit(100.0))) \
                    .withColumn("preschool_deficit", lit(100.0) - coalesce(col("percent_preschool_enrollment"), lit(100.0))) \
                    .withColumn(
                        "Intervention_Priority_Score",
                        (col("child_poverty") * 3.0) + 
                        (col("uninsured") * 2.0) + 
                        (col("high_housing") * 1.5) +
                        (col("hs_deficit") * 1.5) +          # Wayne Foundation Education focus
                        (col("preschool_deficit") * 1.0)     # Wayne Foundation Early Childcare focus
                    )

# 4. Limit to Top 3 Hotspots based on the composite score
top_3_targets_df = scored_df.orderBy(desc("Intervention_Priority_Score")).limit(3)

# 5. Materialize to Delta for lineage and export to Python dictionaries for Gemini
top_3_targets_df.write.mode("overwrite").format("delta").saveAsTable("workspace.wayne_stark.target_cities")
target_cities = top_3_targets_df.toPandas().to_dict(orient="records")

# Verify the output
display(top_3_targets_df)

city,state,median_household_income,percent_unemployed,percent_poverty_level,percent_high_housing_costs,percent_hs_graduates,percent_uninsured,infant_mortality_rate,life_expectancy,total_population,percent_under_18,percent_children_in_poverty,percent_preschool_enrollment,child_poverty,uninsured,high_housing,hs_deficit,preschool_deficit,Intervention_Priority_Score
Detroit,MI,24820.0,27.7,65.3,40.4,78.2,19.6,14.7,73.0,701524.0,25.4,61.4,45.9,61.4,19.6,40.4,21.799999999999997,54.1,370.8
Cleveland,OH,26096.0,19.4,63.0,38.7,78.0,16.7,null,73.6,390923.0,24.2,58.5,42.5,58.5,16.7,38.7,22.0,57.5,357.45
Phoenix,AZ,46601.0,9.7,47.4,31.8,80.6,23.6,7.1,80.0,1537045.0,27.1,38.0,30.9,38.0,23.6,31.8,19.400000000000006,69.1,307.1


## 3. Executive Joint Briefing
**Goal:** Synthesize the 3 localized action plans into a single C-suite deployment memo.

The final LLM pass takes the structured JSON allocations and generates a natural-language executive briefing. By splitting the generation phases (Extraction/Allocation first, Synthesis second), we ensure the AI anchors its final recommendations on the deterministic data we engineered in PySpark.

In [0]:
import json
import time
import random
import requests
import concurrent.futures

# Reusing the exact same API key secret pattern from the Spidey project
API_KEY = dbutils.secrets.get(catalog="workspace", schema="wayne_stark", key="gemini_api_key")

MODEL = "gemini-3.1-flash-lite"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

MAX_ATTEMPTS = 4
REQUEST_TIMEOUT = 60
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

def allocate_city_strategy(city_data):
    context = json.dumps(city_data)
    
    prompt = f"""
    You are the integrated J.A.R.V.I.S. / Batcomputer urban analysis system.
    Analyze the demographic indicators for this municipality: {context}

    Output a structured JSON response assigning strategic resources based on the highest deficits:
    - Stark Industries handles medical infrastructure, clinical access, and high uninsured or infant mortality rates.
    - The Wayne Foundation handles community relief, housing cost burdens, childhood poverty, and education gaps (like high school and preschool deficits).

    Format as strict JSON:
    {{
        "City": "City Name, State",
        "Priority_Score": "Numeric score string rounded to 2 decimals",
        "Wayne_Foundation_Initiative": "1-2 sentence social/educational intervention targeting the specific data",
        "Stark_Industries_Initiative": "1-2 sentence tech/health intervention targeting the specific data",
        "Budget_Split": "e.g., 55% Wayne / 45% Stark based on whether poverty/education or healthcare is more acute"
    }}
    """

    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "responseMimeType": "application/json",
            "temperature": 0.2
        }
    }
    headers = {"Content-Type": "application/json", "x-goog-api-key": API_KEY}

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.post(ENDPOINT, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            if response.status_code == 200:
                raw_text = response.json()["candidates"][0]["content"]["parts"][0]["text"]
                return json.loads(raw_text)
            elif response.status_code in RETRYABLE_STATUS_CODES:
                if attempt == MAX_ATTEMPTS:
                    raise RuntimeError(f"Exhausted retries ({response.status_code}): {response.text}")
                time.sleep((2 ** attempt) + random.uniform(0.1, 0.8))
            else:
                raise RuntimeError(f"Non-retryable HTTP {response.status_code}: {response.text}")
        except requests.RequestException as e:
            if attempt == MAX_ATTEMPTS:
                raise RuntimeError(f"Failed after {MAX_ATTEMPTS} attempts: {e}")
            time.sleep((2 ** attempt) + random.uniform(0.1, 0.8))

print("Engaging Batcomputer / J.A.R.V.I.S. tactical link...")
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    initiative_briefs = list(executor.map(allocate_city_strategy, target_cities))

print(json.dumps(initiative_briefs, indent=2))

Engaging Batcomputer / J.A.R.V.I.S. tactical link...
[
  {
    "City": "Detroit, MI",
    "Priority_Score": "370.80",
    "Wayne_Foundation_Initiative": "Implement comprehensive community development programs focusing on early childhood education and housing subsidies to address the 61.4% child poverty rate and 40.4% housing cost burden. These initiatives will prioritize closing the 54.1% preschool enrollment gap and stabilizing family living conditions.",
    "Stark_Industries_Initiative": "Deploy mobile diagnostic clinics and subsidized health insurance enrollment platforms to mitigate the 19.6% uninsured rate and high infant mortality. Stark-tech infrastructure will be utilized to improve clinical access and health outcomes for the most vulnerable populations.",
    "Budget_Split": "65% Wayne / 35% Stark"
  },
  {
    "City": "Cleveland, OH",
    "Priority_Score": "357.45",
    "Wayne_Foundation_Initiative": "Implement comprehensive community support programs focusing on early child

In [0]:
from IPython.display import Markdown

def generate_joint_briefing(briefs):
    prompt = f"""
    You are the integrated AI representing the J.A.R.V.I.S. and Batcomputer mainframes.
    You are briefing Bruce Wayne and Tony Stark on the deployment of their joint philanthropic venture.
    
    Review the 3 targeted municipal action plans: {json.dumps(briefs)}

    Write a 3-paragraph executive briefing addressed directly to Mr. Wayne and Mr. Stark. 
    1. Detail why Detroit, Cleveland, and Phoenix were selected based on their acute deficits.
    2. Explain how their respective corporate wings (Wayne's social/educational focus vs. Stark's tech/health focus) will complement each other in these cities.
    3. End with a formal request for authorization to begin capital deployment.
    """

    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.7} # Higher temp for better narrative flow
    }
    
    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": API_KEY
    }

    response = requests.post(ENDPOINT, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
    
    if response.status_code == 200:
        return response.json()["candidates"][0]["content"]["parts"][0]["text"]
    else:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text}")

# Generate and display the final memo
final_memo = generate_joint_briefing(initiative_briefs)
display(Markdown(f"### 🦇⚡ Wayne-Stark Initiative: Executive Joint Protocol\n\n{final_memo}"))


### 🦇⚡ Wayne-Stark Initiative: Executive Joint Protocol

Mr. Wayne, Mr. Stark. The integrated mainframe has completed the cross-referencing of socio-economic data across the continental United States. Detroit, Cleveland, and Phoenix have been identified as the primary theaters for our initial deployment due to their critical Priority Scores, which reflect an alarming convergence of systemic instability. Detroit and Cleveland represent the most acute humanitarian crises, characterized by child poverty rates exceeding 58% and severe educational deficits. Phoenix, while distinct in its geography, presents a compounding threat through its staggering 69.1% preschool enrollment deficit and rising infant mortality rates. These municipalities are currently trapped in cycles of structural neglect that require immediate, high-impact intervention.

The operational synergy between your organizations is designed to address both the root causes and the immediate symptoms of these crises. The Wayne Foundation will lead with long-term foundational stability, focusing on housing security and the pedagogical infrastructure necessary to close the preschool enrollment gap. Concurrently, Stark Industries will provide the tactical deployment of mobile diagnostic units and proprietary health-tech platforms to bridge the clinical access divide. By aligning the Wayne Foundation’s community-centric stabilization programs with Stark Industries’ rapid-response health infrastructure, we create a dual-front approach that ensures families are not only housed and educated but also physically capable of sustaining that newfound upward mobility.

The proposed budgetary allocations—averaging a 67/33 split favoring the Wayne Foundation’s social programming—are optimized to maximize the efficacy of these initiatives. All municipal logistics, supply chains, and diagnostic arrays are standing by and synchronized across both the Batcomputer and J.A.R.V.I.S. networks. I am prepared to initiate the first phase of capital deployment and logistical mobilization immediately upon your joint confirmation. Please provide your authorization to proceed.